# 08: Numbers for the conclusion, limitations and future scope (Phase 9)

The conclusion may only use results already shown in earlier phases. This notebook collects every number quoted in `report/drafts/conclusion.md` and `report/drafts/limitations_future_scope.md` in one place, recomputed from the saved tables, so each sentence can be traced. Nothing new is modelled.

In [1]:
import sys
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, roc_auc_score

sys.path.insert(0, "..")
from src import config
from src.preprocess import ITEMS

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
P = config.PROCESSED_DIR
ratios = pd.read_csv(P / "ratios.csv")
analysis = ratios[ratios["analysis_year"]]
sector = pd.read_csv(config.DASHBOARD_EXPORTS_DIR / "sector_by_year.csv")
comparison = pd.read_csv(P / "model_comparison.csv")
test_pred = pd.read_csv(P / "test_predictions.csv")
wf = pd.read_csv(P / "walk_forward_predictions.csv")
importance = pd.read_csv(P / "feature_importance.csv")
balance = pd.read_csv(P / "class_balance.csv")
hyp = pd.read_csv(P / "case_hypotheses.csv")
outcomes = pd.read_csv(P / "case_prediction_outcomes.csv")

## 1. Sector picture (operating companies and all companies)

In [2]:
s = sector.copy()
s["pct_distressed"] = (s["pct_distressed"] * 100).round(1)
s["median_net_margin"] = (s["median_net_margin"] * 100).round(1)
s["loss_share"] = (s["loss_making"] / s["companies"] * 100).round(1)
s.pivot(index="label_year", columns="scope",
        values=["companies", "pct_distressed", "median_current_ratio", "median_net_margin", "loss_share"]).round(2)

companies                                          pct_distressed                                          median_current_ratio                                          median_net_margin  \
scope      All companies Operating companies (revenue >= USD 10m)  All companies Operating companies (revenue >= USD 10m)        All companies Operating companies (revenue >= USD 10m)     All companies   
label_year                                                                                                                                                                                                  
2015                93.0                                     82.0           30.6                                     28.2                 3.40                                     3.57               1.1   
2016                83.0                                     77.0           49.4                                     46.6                 3.55                                     3.80               1.7   
2017                86.0                                     77.0           45.7                                     39.7                 3.40                                     3.65               2.9   
2018                85.0                                     75.0           40.2                                     34.2                 3.32                                     3.39               0.7   
2019                83.0                                     72.0           46.9                                     41.4                 2.81                                     2.82               0.0   
2020                81.0                                     71.0           39.7                                     35.3                 3.18                                     3.15               4.5   
2021                89.0                                     76.0           44.7                                     37.0                 3.13                                     3.04               5.7   
2022                83.0                                     73.0           37.8                                     30.6                 2.98                                     2.92               5.6   
2023                81.0                                     73.0           40.0                                     33.3                 3.21                                     3.21              -1.9   
2024                81.0                                     71.0           51.2                                     44.3                 2.89                                     3.12             -11.7   
2025                80.0                                     72.0           41.0                                     35.7                 2.65                                     2.69              -1.4   

                                                       loss_share                                           
scope      Operating companies (revenue >= USD 10m) All companies Operating companies (revenue >= USD 10m)  
label_year                                                                                                  
2015                                            2.1          49.5                                     43.9  
2016                                            2.0          47.0                                     42.9  
2017                                            6.2          47.7                                     41.6  
2018                                            3.3          48.2                                     42.7  
2019                                            2.0          49.4                                     41.7  
2020                                            6.5          40.7                                     33.8  
2021                                            8.5          42.7                                     32.9  
2022                                            7.7   

In [3]:
op = s[s["scope"].str.startswith("Operating")].set_index("label_year")
print(f"Operating companies, share distressed: min {op.pct_distressed.min()}% ({op.pct_distressed.idxmin()}), "
      f"max {op.pct_distressed.max()}% ({op.pct_distressed.idxmax()}), mean {op.pct_distressed.mean():.1f}%")
print(f"Median current ratio: {op.median_current_ratio[2015]:.2f} (2015) -> {op.median_current_ratio[2025]:.2f} (2025); "
      f"min {op.median_current_ratio.min():.2f} ({op.median_current_ratio.idxmin()})")
print(f"Median net margin: best {op.median_net_margin.max()}% ({op.median_net_margin.idxmax()}), "
      f"worst {op.median_net_margin.min()}% ({op.median_net_margin.idxmin()}), 2025 {op.median_net_margin[2025]}%")
print(f"Loss-making share: {op.loss_share[2015]}% (2015), {op.loss_share.max()}% max ({op.loss_share.idxmax()}), {op.loss_share[2025]}% (2025)")
print("One-year-ahead distressed share (operating, all feature years):",
      f"{balance.loc[balance.feature_year == 'All', 'share_excl_low_revenue'].iloc[0]:.1%}")

Operating companies, share distressed: min 28.2% (2015), max 46.6% (2016), mean 36.9%
Median current ratio: 3.57 (2015) -> 2.69 (2025); min 2.69 (2025)
Median net margin: best 8.5% (2021), worst -2.5% (2024), 2025 1.2%
Loss-making share: 43.9% (2015), 54.9% max (2024), 47.2% (2025)
One-year-ahead distressed share (operating, all feature years): 37.2%


## 2. Case companies: one headline number each

In [4]:
case = analysis[analysis["cik"].isin(config.CASE_COMPANIES)].copy()
case["company"] = case["cik"].map(config.CASE_COMPANIES)
zs = pd.read_csv(P / "zscores.csv")[["cik", "label_year", "z_prime"]]
case = case.merge(zs, on=["cik", "label_year"])
pv = case.pivot(index="label_year", columns="company", values=["revenue", "net_income", "operating_margin", "z_prime"])
pv.loc[[2015, 2019, 2021, 2022, 2023, 2024, 2025]].round(2)

revenue                              net_income                              operating_margin                        z_prime                       
company         AMD    Intel    Nvidia Wolfspeed        AMD    Intel    Nvidia Wolfspeed              AMD Intel Nvidia Wolfspeed     AMD Intel Nvidia Wolfspeed
label_year                                                                                                                                                     
2015         3991.0  55355.0    5010.0    1632.5     -660.0  11420.0     614.0    -64.05            -0.12  0.25   0.15     -0.04   -1.03  2.06   2.52      2.90
2019         6731.0  71965.0   10918.0    1080.0      341.0  21048.0    2796.0   -375.10             0.09  0.31   0.26     -0.01    1.08  1.96   3.37      1.50
2021        16434.0  79024.0   26914.0     525.6     3162.0  19868.0    9752.0   -523.90             0.22  0.25   0.37     -0.60    3.02  1.85   2.66      0.37
2022        23601.0  63054.0   26974.0     746.2     1320.0   8014.0    4368.0   -200.90             0.05  0.04   0.16     -0.33    2.29  1.33   1.96      0.56
2023        22680.0  54228.0   60922.0     921.9      854.0   1689.0   29760.0   -329.90             0.02  0.00   0.54     -0.41    2.42  1.19   4.03      0.16
2024        25785.0  53101.0  130497.0     807.2     1641.0 -18756.0   72880.0   -864.20             0.07 -0.22   0.62     -0.55    2.68  0.79   5.38     -0.12
2025        34639.0  52853.0  215938.0     757.6     4335.0   -267.0  120067.0  -1609.20             0.11 -0.04   0.60     -1.75    2.74  1.09   5.26     -1.55

In [5]:
print(hyp.groupby("company")["verdict"].apply(lambda v: v.str.split(" ").str[0].value_counts().to_dict()).to_string())
k = outcomes[outcomes["label_year"] <= 2024]
print("\nCase-company walk-forward outcomes 2018-2024:", k["outcome"].value_counts().to_dict())

company                
AMD        Supported       3.0
           Partly          NaN
           Contradicted    NaN
Intel      Supported       3.0
           Partly          2.0
           Contradicted    NaN
Nvidia     Supported       2.0
           Partly          1.0
           Contradicted    1.0
Wolfspeed  Supported       4.0
           Partly          NaN
           Contradicted    1.0

Case-company walk-forward outcomes 2018-2024: {'correct (healthy)': 17, 'correct warning': 8, 'missed': 2, 'false alarm': 1}


## 3. Algorithm comparison

In [6]:
t = comparison[comparison["set"] == "test"].set_index("model")[["precision", "recall", "f1", "roc_auc", "pr_auc", "threshold"]]
v = comparison[comparison["set"] == "validation"].set_index("model")[["f1", "roc_auc", "pr_auc"]].add_prefix("val_")
table = t.join(v).round(3)
table

,precision,recall,f1,roc_auc,pr_auc,threshold,val_f1,val_roc_auc,val_pr_auc
model,,,,,,,,,
Logistic regression,0.826,0.704,0.760,0.885,0.844,0.770,0.843,0.920,0.849
Decision tree,0.796,0.722,0.757,0.852,0.756,0.933,0.811,0.913,0.860
Random forest,0.746,0.815,0.779,0.895,0.883,0.494,0.835,0.942,0.917
XGBoost,0.812,0.722,0.765,0.894,0.878,0.721,0.795,0.942,0.902
Baseline: current ratio < 1,1.000,0.074,0.138,0.756,0.715,NaN,0.128,0.700,0.579
Baseline: persistence (Z' < 1.23 now),0.880,0.815,0.846,0.953,0.939,NaN,0.847,0.926,0.910


In [7]:
rf, cr, ps = table.loc["Random forest"], table.loc["Baseline: current ratio < 1"], table.loc["Baseline: persistence (Z' < 1.23 now)"]
for m in ["f1", "roc_auc", "pr_auc", "recall"]:
    print(f"{m:8s} RF {rf[m]:.3f} | vs current-ratio rule {rf[m] - cr[m]:+.3f} | vs persistence {rf[m] - ps[m]:+.3f}")
models = ["Logistic regression", "Decision tree", "Random forest", "XGBoost"]
print("\nSpread of test F1 across the four models:", f"{table.loc[models, 'f1'].min():.3f}-{table.loc[models, 'f1'].max():.3f}")
print("Spread of test ROC-AUC:", f"{table.loc[models, 'roc_auc'].min():.3f}-{table.loc[models, 'roc_auc'].max():.3f}")

f1       RF 0.779 | vs current-ratio rule +0.641 | vs persistence -0.067
roc_auc  RF 0.895 | vs current-ratio rule +0.139 | vs persistence -0.058
pr_auc   RF 0.883 | vs current-ratio rule +0.168 | vs persistence -0.056
recall   RF 0.815 | vs current-ratio rule +0.741 | vs persistence +0.000

Spread of test F1 across the four models: 0.757-0.779
Spread of test ROC-AUC: 0.852-0.895


In [8]:
# Early warnings: test rows healthy at t (Z' >= 1.23) that became distressed at t+1.
thr = t["threshold"]
trans = test_pred[(test_pred["distressed"] == 0) & (test_pred["distressed_t1"] == 1)]
print("Healthy -> distressed transitions in the test years:", len(trans))
for m in models:
    print(f"  {m:20s} flagged {int((trans[f'prob_{m}'] >= thr[m]).sum())} of {len(trans)}")

Healthy -> distressed transitions in the test years: 10
  Logistic regression  flagged 2 of 10
  Decision tree        flagged 1 of 10
  Random forest        flagged 3 of 10
  XGBoost              flagged 3 of 10


In [9]:
# Walk-forward (out of sample, all scored operating companies with a known outcome)
w = wf[wf["distressed_t1"].notna()]
print(f"Walk-forward rows: {len(w)}, ROC-AUC {roc_auc_score(w.distressed_t1, w.prob_distressed_t1):.3f}, "
      f"F1 {f1_score(w.distressed_t1, w.pred_distressed_t1):.3f}")
imp = importance.sort_values("shap_share", ascending=False)
print("\nTop five by SHAP share:", ", ".join(f"{f} {s:.1%}" for f, s in zip(imp.feature, imp.shap_share.head(5))))
print("Top three together:", f"{imp.shap_share.head(3).sum():.1%}")
print("Top five by permutation importance:", ", ".join(importance.sort_values('perm_importance_mean', ascending=False).feature.head(5)))

Walk-forward rows: 469, ROC-AUC 0.922, F1 0.800

Top five by SHAP share: interest_coverage 13.5%, operating_margin 13.3%, net_margin 12.0%, roa 11.1%, roe 8.5%
Top three together: 38.8%
Top five by permutation importance: operating_margin, debt_to_equity, debt_ratio, log_total_assets, net_margin


## 4. Numbers that size the limitations

In [10]:
# Share of statement values that came from a fallback tag (not the first tag in the priority list).
tags = pd.read_csv(P / "tag_sources.csv")
tags = tags[tags["label_year"].between(config.LABEL_YEAR_START, config.LABEL_YEAR_END)]
first = {item: tl[0] for item, (_, tl) in ITEMS.items()}
tags["fallback"] = tags["source"] != tags["item"].map(first)
fb = tags.groupby("item")["fallback"].agg(["sum", "size"])
fb["share"] = (fb["sum"] / fb["size"] * 100).round(1)
print(f"All items: {tags.fallback.sum()} of {len(tags)} values from a fallback tag ({tags.fallback.mean():.1%})")
fb.sort_values("share", ascending=False).head(8)

All items: 2485 of 16442 values from a fallback tag (15.1%)


,sum,size,share
item,,,
temporary_equity,44,47,93.6
revenue,639,925,69.1
cost_of_revenue,499,890,56.1
interest_expense,467,880,53.1
pretax_income,295,842,35.0
interest_paid,135,592,22.8
total_liabilities,165,925,17.8
inventory,66,905,7.3


In [11]:
flags = ["total_liabilities_derived", "coverage_uses_proxy", "inventory_assumed_zero", "gross_profit_derived",
         "net_income_from_profitloss", "equity_includes_nci", "cash_includes_restricted"]
print("Company-years in analysis years:", len(analysis))
print(analysis[flags].sum().to_string())
print("\nInterest source:", analysis["interest_source"].value_counts().to_dict())
print("\nFiscal year-end months of all companies (analysis years):")
print(pd.to_datetime(analysis["fiscal_year_end"]).dt.month.value_counts().sort_index().to_dict())

Company-years in analysis years: 925
total_liabilities_derived     165
coverage_uses_proxy           165
inventory_assumed_zero         66
gross_profit_derived           52
net_income_from_profitloss     38
equity_includes_nci            26
cash_includes_restricted       24

Interest source: {'reported': 509, 'cash_paid_proxy': 181, 'zero_no_debt': 174, 'not_available': 45, 'companyfacts': 16}

Fiscal year-end months of all companies (analysis years):
{1: 47, 2: 10, 3: 79, 4: 9, 6: 78, 8: 31, 9: 55, 10: 45, 11: 8, 12: 563}


In [12]:
sizes = comparison[comparison["model"] == "Random forest"].set_index("set")[["rows", "distressed"]]
print(sizes)
print("\nRobustness label positives (one-year-ahead):", int(balance.loc[balance.feature_year == "All", "robust_t1_positive"].iloc[0]))
print("Grey-zone company-years that would flip to distressed if the grey zone counted:",
      int(balance.loc[balance.feature_year == "All", "grey_t1_would_flip"].iloc[0]))

            rows  distressed
set                         
validation   139          43
test         134          54

Robustness label positives (one-year-ahead): 29
Grey-zone company-years that would flip to distressed if the grey zone counted: 266
